In [ ]:
from platform import python_version
print(python_version())

In [ ]:
!echo $CONDA_DEFAULT_ENV

### Install Cellpose-SAM

https://pypi.org/project/cellpose/

https://github.com/MouseLand/cellpose/blob/main/notebooks/run_Cellpose-SAM.ipynb

In [ ]:
!nvidia-smi

In [ ]:
!nvcc --version

In [ ]:
# !pip3 install scikit-learn
# !pip3 install seaborn
# !pip3 install plotly

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(1, '../src/')

from Basic import *
from image_lib import *

#### Check if colab notebook instance has GPU access

In [ ]:
root0 = "../../colaboracoes/deOcesano"
 
os.listdir(root0)

In [ ]:
cp = Cellpose(root0=root0, verbose=True)

In [ ]:
# !pip3 install pyyaml
cp.set_default_parameters(root_yaml=os.getcwd(), verbose=True)
# cp.dic_yml

### Root where the images (samples) are

In [ ]:
cp.root_samples

### Root where the crop images (ncropxncrop) are

In [ ]:
cp.root_crop

### Plate: root_plate (images), root_tabel_image (dir dual)

In [ ]:
plates = cp.list_plates(s_start='Plate')
plates

In [ ]:
i=0
plate=plates[i]

cp.set_plate_params(plate=plate, verbose=True)

In [ ]:
cp.experiments

In [ ]:
probes = cp.probes
probes

### Experiments or Samples or Images

In [ ]:
j=2
experiment = cp.experiments[j]
cp.create_roots_experiment(experiment, verbose=True)

# print(f"\n\nexperiment: '{cp.experiment}'  probe='{cp.probe}' {'is ok' if cp.probe in cp.probes else 'error'}")

### Images available

In [ ]:
fnames = cp.list_images_already_set(image_type='tif', verbose=True)
len(fnames), fnames[0]

### Get one image

In [ ]:
i=0
fname=fnames[i]
img = cp.read_PIL_image(fname, cp.root_image, verbose=True)
(width, height) = img.size
print(width, height)

In [ ]:
cp.display_img(img, figsize=(5,5));

### Get one image

In [ ]:
i=0
fname=fnames[i]
img = cp.read_PIL_image(fname, cp.root_image, verbose=True)
(width, height) = img.size
print(width, height)

In [ ]:
cp.display_img(img, figsize=(5,5));

### Crop dirs

In [ ]:
cp.root_crop, cp.root_image

In [ ]:
cp.plate, cp.class_names

In [ ]:
root_data, dic_root_train, dic_root_test = \
cp.set_data_origin_and_create_roots_to_train_and_test(image_example=img,verbose=True)

In [ ]:
print(cp.width, cp.height)
print(cp.size_x, cp.size_y)

In [ ]:
print(cp.model_fname)
print(cp.model_table)

In [ ]:
cp.clean_train_and_test(verbose=True)

### Create crop image ensemble

In [ ]:
force=False
verbose=False

plates = cp.list_plates(s_start='Plate')

for plate in plates:
    cp.set_plate_params(plate, verbose=False)
    print(f">>> plate {cp.plate}")

    for experiment in cp.experiments:
        cp.create_roots_experiment(experiment, verbose=False)

        fname_imgs = cp.list_images_already_set(image_type='tif', verbose=False)

        img = cp.read_PIL_image(fname_imgs[0], cp.root_image, verbose=False)
        root_data, dic_root_train, dic_root_test = \
        cp.set_data_origin_and_create_roots_to_train_and_test(image_example=img,verbose=False)

        print(f"\texperiment {cp.experiment:40} has {len(fname_imgs):3} images.")            

In [ ]:
force=False
verbose=False

plates = cp.list_plates(s_start='Plate')

for plate in plates:
    cp.set_plate_params(plate, verbose=False)
    print(f">>> plate {cp.plate}")
    
    for experiment in cp.experiments:
        cp.create_roots_experiment(experiment, verbose=False)

        fname_imgs = cp.list_images_already_set(image_type='tif', verbose=False)

        img = cp.read_PIL_image(fname_imgs[0], cp.root_image, verbose=False)
        root_data, dic_root_train, dic_root_test = \
        cp.set_data_origin_and_create_roots_to_train_and_test(image_example=img,verbose=False)

        print(f"\texperiment {cp.experiment:40} has {len(fname_imgs):3} images.")

        for fname in fname_imgs:
            for ncrop in [5, 10, 20]:
                _ = cp.crop_squares_already_set(fname, ncrop, force=force, verbose=verbose)
                print('.',end='')
        print('')


### Display cropped

In [ ]:
figsize=(8,8)

force=False
verbose=False

plates = cp.list_plates(s_start='Plate')

i=0
plate=plates[i]
cp.create_roots_plate(plate, verbose=False)

experiments = cp.list_experiments(flg_is_dir=True)
j=0
experiment=experiments[j]
cp.create_roots_experiment(experiment, verbose=False)

files = cp.list_images_already_set(image_type='tif', verbose=False)
k=1
fname=files[k]

print(f">>> plate {cp.plate} experiment {cp.experiment} has {len(files)} images:")
print(f"image {k} '{fname}'")
print("")

m=0
ncrop = 5

df = cp.crop_squares_already_set(fname, lossless=True, image_type='png', force=force, verbose=verbose)
if df is not None and not df.empty:
    fig, axes = cp.display_cropped_img_from_df(df, transpose=False, figsize=figsize)